In [0]:
import mlflow
import mlflow.pyfunc
import pandas as pd
import numpy as np

from pyspark.sql.functions import max

from pyspark.ml.feature import (
    StringIndexerModel,
    OneHotEncoderModel,
    VectorAssembler
)

In [0]:
model_uri = "models:/Readmission_XGBoost_Model@Champion"

model = mlflow.pyfunc.load_model(model_uri)

In [0]:
gold_df = spark.table(
    "db_healthcare_kl.gold.ml_readmission_features"
)

In [0]:
latest_time = (
    gold_df
    .select(max("processed_timestamp"))
    .collect()[0][0]
)

new_patients = gold_df.filter(
    gold_df.processed_timestamp == latest_time
)

In [0]:
latest_time = (
    gold_df
    .select(max("processed_timestamp"))
    .collect()[0][0]
)

new_patients = gold_df.filter(
    gold_df.processed_timestamp == latest_time
)

In [0]:
feature_df = new_patients.drop(
    "person_id",
    "processed_timestamp",
    "readmission_30day_flag"
)

In [0]:
visit_indexer = StringIndexerModel.load(
    "/Volumes/db_healthcare_kl/default/ml_models/visit_indexer"
)

diagnosis_indexer = StringIndexerModel.load(
    "/Volumes/db_healthcare_kl/default/ml_models/diagnosis_indexer"
)

cohort_indexer = StringIndexerModel.load(
    "/Volumes/db_healthcare_kl/default/ml_models/cohort_indexer"
)

In [0]:
feature_df = visit_indexer.transform(feature_df)
feature_df = diagnosis_indexer.transform(feature_df)
feature_df = cohort_indexer.transform(feature_df)

In [0]:
visit_encoder = OneHotEncoderModel.load(
    "/Volumes/db_healthcare_kl/default/ml_models/visit_encoder"
)

diagnosis_encoder = OneHotEncoderModel.load(
    "/Volumes/db_healthcare_kl/default/ml_models/diagnosis_encoder"
)

cohort_encoder = OneHotEncoderModel.load(
    "/Volumes/db_healthcare_kl/default/ml_models/cohort_encoder"
)

In [0]:
feature_df = visit_encoder.transform(feature_df)
feature_df = diagnosis_encoder.transform(feature_df)
feature_df = cohort_encoder.transform(feature_df)

In [0]:
assembler = VectorAssembler.load(
    "/Volumes/db_healthcare_kl/default/ml_models/assembler"
)

In [0]:
feature_df = assembler.transform(feature_df)

In [0]:
pdf = feature_df.select("features").toPandas()

In [0]:
feature_names = [f"feature_{i}" for i in range(46)]

X = pd.DataFrame(
    np.vstack(
        pdf["features"].apply(lambda x: x.toArray())
    ),
    columns=feature_names
)

In [0]:
predictions = model.predict(X)

In [0]:
result = new_patients.toPandas()

result["prediction"] = predictions

In [0]:
prediction_df = spark.createDataFrame(result)

In [0]:
prediction_df.write \
    .mode("overwrite") \
    .saveAsTable(
        "db_healthcare_kl.gold.readmission_predictions"
    )

In [0]:
from pyspark.sql.functions import when

prediction_df = prediction_df.withColumn(
    "risk",
    when(prediction_df.prediction == 1, "High Risk")
    .otherwise("Low Risk")
)

display(prediction_df)